# Implement SCD Type 2 Customer History

## Business Scenario

A retail company maintains a **Customer Dimension** table.

When customer information changes (address, city, email, phone, segment, etc.), the business wants to:

- ✅ Preserve historical records
- ✅ Track when a value changed
- ✅ Know what values were valid at a specific point in time
- ✅ Support audit and compliance requirements

Instead of overwriting old data, a **new version** of the customer record is created.

This is called **Slowly Changing Dimension Type 2 (SCD2)**.

## Existing Target Table

**Current Customer Dimension:**

| customer_id | customer_name | city   | segment | start_date  | end_date    | is_current |
|-------------|---------------|--------|---------|-------------|-------------|------------|
| 101         | Amit          | Delhi  | Silver  | 2026-01-01  | 9999-12-31  | Y          |
| 102         | Rahul         | Mumbai | Silver  | 2026-01-01  | 9999-12-31  | Y          |

**Notice:**

- Customer **101's** segment changed: `Silver -> Gold`
- Customer **103 (Priya)** is a new customer

## Expected Result After SCD2

| customer_id | customer_name | city   | segment | start_date  | end_date    | is_current |
|-------------|---------------|--------|---------|-------------|-------------|------------|
| 101         | Amit          | Delhi  | Silver  | 2026-01-01  | 2026-11-17  | N          |
| 101         | Amit          | Delhi  | Gold    | 2026-11-18  | 9999-12-31  | Y          |
| 102         | Rahul         | Mumbai | Silver  | 2026-01-01  | 9999-12-31  | Y          |
| 103         | Priya         | Pune   | Gold    | 2026-11-18  | 9999-12-31  | Y          |

In [0]:
%sql
USE Catalog Databricks_learning;
USE Schema concepts;

In [0]:
from pyspark.sql import functions as F

customer_dim_data = [
    (101, "Amit",  "Delhi",  "Silver", "2024-01-01", "9999-12-31", "Y"),
    (102, "Rahul", "Mumbai", "Gold",   "2024-01-01", "9999-12-31", "Y")
]

customer_dim = spark.createDataFrame(
    customer_dim_data,
    [
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "effective_date",
        "expiry_date",
        "is_current"
    ]
)

(customer_dim
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("customer_dim"))

target_df = spark.table("customer_dim")
target_df.display()

In [0]:
source_data = [
    # Existing customer changed segment
    (101, "Amit", "Delhi", "Platinum"),
    # Existing customer unchanged
    (102, "Rahul", "Mumbai", "Gold"),
    # New customer
    (103, "Priya", "Pune", "Silver")
]
source_df = spark.createDataFrame(
    source_data,
    [
        "customer_id",
        "customer_name",
        "city",
        "segment"
    ]
)

source_df.display()

In [0]:
changed_records = (
    source_df.alias("src")
    .join(
        target_df.alias("tgt"),
        "customer_id"
    )
    .where(
        (F.col("tgt.is_current") == "Y")
        &
        (
            (F.col("src.city") != F.col("tgt.city"))
            |
            (F.col("src.segment") != F.col("tgt.segment"))
        )
    )
    .select(
        "customer_id",
        F.col("src.customer_name"),
        F.col("src.city"),
        F.col("src.segment")
    )
    .cache() #  lazy evaluation plus the data changing between the two runs 
)

display(changed_records)

In [0]:
display(spark.table("customer_dim"))

In [0]:
display(changed_records)

In [0]:
new_versions = (
    changed_records
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment"
    )
    .withColumn(
        "effective_date",
        F.current_date()
    )
    .withColumn(
        "expiry_date",
        F.lit("9999-12-31")
    )
    .withColumn(
        "is_current",
        F.lit("Y")
    )
)
new_versions.display()

In [0]:
new_customers = (
    source_df.alias("src")
    .join(
        target_df.alias("tgt"),
        "customer_id",
        "left_anti"
    )
    .withColumn(
        "effective_date",
        F.current_date()
    )
    .withColumn(
        "expiry_date",
        F.lit("9999-12-31")
    )
    .withColumn(
        "is_current",
        F.lit("Y")
    )
)

In [0]:
new_customers.display()

In [0]:
final_insert_df = (
    new_versions.unionByName(new_customers)
)
final_insert_df.show()

In [0]:
from delta.tables import DeltaTable

delta_tbl = DeltaTable.forName(
    spark,
    "customer_dim"
)

delta_tbl.alias("tgt").merge(
    changed_records.alias("src"),
    """
    tgt.customer_id = src.customer_id
    AND tgt.is_current='Y'
    """
).whenMatchedUpdate(
    set={
        "expiry_date": "current_date()-1",
        "is_current": "'N'"
    }
).execute()


In [0]:
display(spark.table("customer_dim"))

In [0]:
final_insert_df.write\
 .format("delta")\
 .mode("append")\
 .saveAsTable("customer_dim")

In [0]:
display(
    spark.table("customer_dim")
    .orderBy(
        "customer_id",
        "effective_date"
    )
)